# BUSI · обучение в Google Colab на T4

В меню **Среда выполнения → Сменить среду выполнения** выберите T4. Ноутбук скачивает изображения BUSI из Kaggle. Укажите папку результатов относительно «Моего диска». Выберите одну модель и seed, затем выполните все ячейки. Для девяти экспериментов повторите запуск с каждой комбинацией. Используйте только одну активную сессию для каталога результатов.

После разрыва сессии выберите те же параметры и режим **resume**. Для короткой проверки укажите одну эпоху на сессию, затем продолжите с лимитом 0. Для итогового test после завершения обучения выберите **test**. Используйте только свои checkpoints: они содержат сериализованное состояние Python.

In [5]:
#@title Параметры запуска
model = "baseline" #@param ["baseline", "resnet18", "vit_tiny"]
seed = 42 #@param [42, 43, 44] {type:"raw"}
mode = "new" #@param ["new", "resume", "test"]
results_folder = "colab/busi-experiments" #@param {type:"string"}
epochs_this_session = 0 #@param {type:"integer"}
assert seed in (42, 43, 44) and epochs_this_session >= 0

In [6]:
from google.colab import drive
from pathlib import Path
import torch

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name():
    raise RuntimeError("Выберите GPU T4 в настройках среды выполнения Colab.")
print(torch.cuda.get_device_name(), torch.__version__, torch.version.cuda)
drive.mount("/content/drive")
%pip -q install timm==1.0.22 kagglehub

import hashlib
import json
import math
import os
import random
import kagglehub
import time
from importlib.metadata import version
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import numpy as np
import pandas as pd
import timm
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, recall_score, roc_auc_score)
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

drive_root = Path("/content/drive/MyDrive")
CLASSES = ("normal", "benign", "malignant")

versions = {p: version(p) for p in ("torch", "torchvision", "timm", "scikit-learn")}
print(versions)
configs = {
    "baseline": dict(batch=32, lr=1e-3, decay=1e-4, epochs=50, patience=8),
    "resnet18": dict(batch=32, lr=1e-3, decay=1e-4, epochs=30, patience=6,
                     head_epochs=5, fine_lr=1e-4),
    "vit_tiny": dict(batch=16, lr=5e-5, decay=0.05, epochs=30, patience=6, warmup=2),
}
cfg = configs[model].copy()
base = drive_root / results_folder
run = base / "runs" / model / f"seed-{seed}"
run.mkdir(parents=True, exist_ok=True)
last, best = run / "last.pt", run / "best.pt"
if mode == "new" and any(run.iterdir()):
    raise ValueError("Каталог занят. Выберите resume или новый каталог результатов.")
if mode != "new" and not last.is_file():
    raise FileNotFoundError(last)

def save_json(value, path):
    tmp = path.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

def save_pt(value, path):
    tmp = path.with_suffix(".tmp.pt")
    torch.save(value, tmp)
    os.replace(tmp, path)

def load_pt(path):
    return torch.load(path, map_location="cpu", weights_only=False)

Tesla T4 2.14.1+cu130 13.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'timm': '1.0.22', 'scikit-learn': '1.6.1'}


## BUSI из Kaggle и общий split

Сессия Colab получает [Breast Ultrasound Images Dataset](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset) через KaggleHub. В классификацию попадают только снимки. Одинаковые PNG с конфликтующими метками исключаются, а для копий с одной меткой сохраняется один путь. Разбиение по классам воспроизводит split_v2 проекта независимо от seed обучения. CSV и его SHA-256 хранятся вместе с результатами на Drive.

In [7]:
download = Path(kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset"))
roots = [p.parent for p in download.rglob("benign")
         if all((p.parent / label).is_dir() for label in CLASSES)]
if len(roots) != 1:
    raise ValueError(f"Ожидался один каталог BUSI, найдены: {roots}")
root = roots[0]

def make_split(root):
    samples = []
    for label in CLASSES:
        for path in sorted((root / label).glob("*.png")):
            if "_mask" in path.stem.lower():
                continue
            samples.append((path.relative_to(root).as_posix(), label,
                            hashlib.sha256(path.read_bytes()).hexdigest()))
    scans = pd.DataFrame(samples, columns=["filepath", "class", "sha256"])
    single_label = scans.groupby("sha256")["class"].transform("nunique") == 1
    unique = scans[single_label].drop_duplicates("sha256")
    rng = random.Random(42)
    records = []
    for label in CLASSES:
        paths = sorted(unique.loc[unique["class"] == label, "filepath"])
        rng.shuffle(paths)
        a = round(len(paths) * 0.70)
        b = a + round(len(paths) * 0.15)
        for part, group in (("train", paths[:a]), ("validation", paths[a:b]), ("test", paths[b:])):
            records.extend((path, label, part) for path in group)
    return pd.DataFrame(records, columns=["filepath", "class", "split"])

rows = make_split(root)
content = rows.to_csv(index=False, lineterminator="\n").encode()
digest = hashlib.sha256(content).hexdigest()
if digest != "5042084b87743cadaabd42319e03f7fbe87fb2976995039d1daf596217aa8ed7":
    raise ValueError("Скачанные данные не воспроизводят эталонное разбиение BUSI split_v2.")
split = base / "splits/split_v2.csv"
split.parent.mkdir(parents=True, exist_ok=True)
for path, data in ((split, content), (split.with_suffix(".sha256"), (digest + "\n").encode())):
    if path.exists() and path.read_bytes().strip() != data.strip():
        raise ValueError(f"Сохранённое разбиение отличается: {path}")
    if not path.exists():
        tmp = path.with_suffix(path.suffix + ".tmp")
        tmp.write_bytes(data)
        os.replace(tmp, path)
# Частоты берём только из train и выстраиваем в том же порядке, что и логиты.
counts = rows.loc[rows.split == "train", "class"].value_counts().reindex(CLASSES)
assert counts.notna().all() and (counts > 0).all()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(1 / counts.to_numpy(), dtype=torch.float32, device="cuda"))

Using Colab cache for faster access to the 'breast-ultrasound-images-dataset' dataset.


In [8]:
class ScanDataset(Dataset):
    def __init__(self, root, subset, transform):
        self.root, self.transform = root, transform
        self.samples = list(zip(subset.filepath, subset["class"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(self.root / path) as image:
            return self.transform(image.convert("RGB")), CLASSES.index(label)

def train_statistics(root, rows):
    # Параметры baseline вычисляются по пикселям train до аугментаций.
    total, squares, count = np.zeros(3), np.zeros(3), 0
    for path in rows.loc[rows.split == "train", "filepath"]:
        with Image.open(root / path) as image:
            rgb = image.convert("RGB").resize((224, 224), Image.Resampling.BILINEAR)
            pixels = np.asarray(rgb, dtype=np.float64).reshape(-1, 3) / 255
        total += pixels.sum(axis=0)
        squares += (pixels * pixels).sum(axis=0)
        count += len(pixels)
    mean = total / count
    std = np.sqrt(np.maximum(squares / count - mean**2, 0))
    return mean.tolist(), np.where(std > 0, std, 1).tolist()

def make_batches(root, rows, mean, std, batch_size, seed):
    aug = [T.RandomHorizontalFlip(), T.RandomRotation(10, interpolation=T.InterpolationMode.BILINEAR),
           T.RandomResizedCrop(224, scale=(0.85, 1), ratio=(1, 1)),
           T.ColorJitter(brightness=0.1, contrast=0.1)]
    batches = {}
    for part in ("train", "validation", "test"):
        train = part == "train"
        transform = T.Compose([T.Resize((224, 224)), *(aug if train else []),
                               T.ToTensor(), T.Normalize(mean, std)])
        dataset = ScanDataset(root, rows[rows.split == part], transform)
        batches[part] = DataLoader(dataset, batch_size=batch_size, shuffle=train,
                                   generator=torch.Generator().manual_seed(seed))
    return batches

## Baseline CNN

Каждый блок свёрток извлекает признаки и уменьшает пространственное разрешение. Глобальное усреднение сводит каждую карту признаков к одному числу. Последний слой возвращает три логита в порядке `normal`, `benign`, `malignant`.

In [9]:
class BaselineCNN(nn.Sequential):
    """Четыре свёрточных блока и классификатор на три класса BUSI."""

    def __init__(self):
        layers = []
        channels = 3
        for width in (32, 64, 128, 256):
            layers.extend([nn.Conv2d(channels, width, 3, padding=1, bias=False),
                           nn.BatchNorm2d(width), nn.ReLU(), nn.MaxPool2d(2)])
            channels = width
        # Последний слой возвращает логиты для CrossEntropyLoss.
        super().__init__(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, 3))

## ResNet-18

Первые пять эпох обучается только новый классификатор. С шестой эпохи обновляется вся сеть с меньшей скоростью обучения. Для нормализации снимков используются параметры тех же весов ImageNet, с которых начинается обучение.

In [10]:
class TransferResNet18(nn.Module):
    """ResNet-18 с головой BUSI и двумя этапами fine-tuning."""

    def __init__(self, pretrained=True):
        super().__init__()
        self.network = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.network.fc = nn.Linear(self.network.fc.in_features, 3)

    def forward(self, images):
        return self.network(images)

    def set_train_phase(self, head_only):
        self.train()
        self.network.requires_grad_(not head_only)
        if head_only:
            # eval() также замораживает статистики BatchNorm во время обучения головы.
            self.network.eval()
            self.network.fc.train().requires_grad_(True)

## ViT Tiny

Для Tiny ViT вход имеет размер 224 × 224. Пробный шаг на отдельной модели проверяет обновление весов и измеряет пиковую память. Выбранный физический размер батча сохраняется в `vit_batch.json` на Drive для всех трёх seed.

In [11]:
class TinyViT(nn.Module):
    """Vision Transformer с входом 224×224 и тремя выходными логитами."""

    weights_id = "vit_tiny_patch16_224.augreg_in21k_ft_in1k"

    def __init__(self, pretrained=True):
        super().__init__()
        self.network = timm.create_model(self.weights_id, pretrained=pretrained, num_classes=3)

    def forward(self, images):
        return self.network(images)

def build(pretrained):
    if model == "resnet18":
        return TransferResNet18(pretrained)
    if model == "vit_tiny":
        return TinyViT(pretrained)
    return BaselineCNN()

def checkpoint_network(net):
    # Имена параметров в best.pt и last.pt совпадают с прежними checkpoint.
    return net.network if isinstance(net, (TransferResNet18, TinyViT)) else net

def set_train_phase(net, epoch):
    if isinstance(net, TransferResNet18):
        net.set_train_phase(head_only=epoch <= cfg["head_epochs"])
    else:
        net.train()

def seed_all():
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def probe(batch):
    # Отдельная модель: пробный шаг не меняет начальные веса эксперимента.
    net = build(False).cuda()
    set_train_phase(net, 1)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg["lr"])
    scale = torch.amp.GradScaler("cuda", enabled=model == "vit_tiny")
    head = next(m for m in reversed(list(net.modules())) if isinstance(m, nn.Linear))
    before = head.weight.detach().clone()
    x = torch.randn(batch, 3, 224, 224, device="cuda")
    labels = torch.arange(batch, device="cuda") % 3
    with torch.autocast("cuda", enabled=model == "vit_tiny"):
        logits = net(x)
        loss = criterion(logits, labels)
    assert logits.shape == (batch, 3) and torch.isfinite(loss)
    scale.scale(loss).backward()
    scale.step(opt)
    scale.update()
    assert not torch.equal(before, head.weight), "Пробный шаг не изменил веса"
    return torch.cuda.max_memory_allocated() / 2**20

policy = base / "vit_batch.json"
if model == "vit_tiny" and policy.exists():
    cfg["batch"] = json.loads(policy.read_text())["batch"]
if mode == "new":
    seed_all()
    torch.cuda.reset_peak_memory_stats()
    retry = False
    try:
        peak = probe(cfg["batch"])
    except torch.cuda.OutOfMemoryError:
        if model != "vit_tiny" or policy.exists() or cfg["batch"] != 16:
            raise
        retry = True
    if retry:
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        cfg["batch"] = 8
        peak = probe(8)
    if model == "vit_tiny" and not policy.exists():
        save_json({"batch": cfg["batch"], "probe_peak_MiB": peak}, policy)
    print("Пробный шаг, пиковая память (МиБ):", round(peak))
# При физическом батче 8 два шага накопления дают ViT эффективный батч 16.
cfg["accumulate"] = 16 // cfg["batch"] if model == "vit_tiny" else 1

Пробный шаг, пиковая память (МиБ): 1107


## Подготовка запуска и восстановление

При продолжении сравниваются параметры запуска, версии библиотек и контрольная сумма split. Затем восстанавливаются веса, оптимизатор, расписание, AMP и генераторы случайных чисел. Следующей будет эпоха после последней записи в `history`.

In [12]:
seed_all()
state = load_pt(last) if mode != "new" else None
net = build(pretrained=state is None).cuda()
if model == "resnet18":
    preset = ResNet18_Weights.DEFAULT.transforms()
    mean, std = preset.mean, preset.std
    weights_id = str(ResNet18_Weights.DEFAULT)
elif model == "vit_tiny":
    mean, std = net.network.pretrained_cfg["mean"], net.network.pretrained_cfg["std"]
    weights_id = net.weights_id
else:
    mean, std = train_statistics(root, rows)
    weights_id = "random"

batches = make_batches(root, rows, mean, std, cfg["batch"], seed)
preprocessing = dict(size=[224, 224], channels="RGB", interpolation="bilinear",
                     mean=list(mean), std=list(std), classes=list(CLASSES), split_sha256=digest)
params = dict(model=model, seed=seed, config=cfg, weights=weights_id,
              preprocessing=preprocessing, versions=versions)
if state and state["params"] != params:
    raise ValueError("Параметры, версии библиотек или split отличаются от checkpoint.")

opt = torch.optim.AdamW(net.parameters(), lr=cfg["lr"], weight_decay=cfg["decay"])
def lr_factor(i):
    if model == "resnet18" and i >= cfg["head_epochs"]:
        return cfg["fine_lr"] / cfg["lr"]
    if model == "vit_tiny":
        warm = cfg["warmup"]
        if i < warm:
            return (i + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (i-warm) / (cfg["epochs"]-warm)))
    return 1.0

schedule = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=model == "vit_tiny")
def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                cpu=torch.get_rng_state(), cuda=torch.cuda.get_rng_state(),
                loaders={k: v.generator.get_state() for k, v in batches.items()})

history, high, best_epoch, stale = [], -1.0, 0, 0
if state:
    checkpoint_network(net).load_state_dict(state["net"])
    opt.load_state_dict(state["optimizer"])
    schedule.load_state_dict(state["schedule"])
    scaler.load_state_dict(state["scaler"])
    history, high = state["history"], state["best_f1"]
    best_epoch, stale = state["best_epoch"], state["stale"]
    rng = state["rng"]
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["cpu"])
    torch.cuda.set_rng_state(rng["cuda"])
    # У каждого загрузчика свой генератор; восстановление train сохраняет порядок батчей.
    for k, v in batches.items():
        v.generator.set_state(rng["loaders"][k])
    # Завершить запись best, если среда отключилась сразу после записи last.
    if best_epoch == len(history):
        save_pt(dict(net=state["net"], params=params, epoch=best_epoch, f1=high), best)
    save_json(history, run / "history.json")
    del state
else:
    save_json(params, run / "params.json")

@torch.inference_mode()
def evaluate(part):
    net.eval()
    ys, scores = [], []
    for x, y in batches[part]:
        logits = net(x.cuda())
        scores.append(logits.float().softmax(1).cpu().numpy())
        ys.extend(y.tolist())
    probs = np.concatenate(scores)
    pred = probs.argmax(1)
    return dict(accuracy=float(accuracy_score(ys, pred)),
                macro_f1=float(f1_score(ys, pred, average="macro")),
                recall_malignant=float(recall_score(ys, pred, labels=[CLASSES.index("malignant")],
                                                    average=None, zero_division=0)[0]),
                roc_auc=float(roc_auc_score(ys, probs, multi_class="ovr", average="macro")),
                confusion=confusion_matrix(ys, pred, labels=range(3)).tolist(),
                per_class=classification_report(ys, pred, labels=range(3), target_names=CLASSES,
                                                output_dict=True, zero_division=0))

## Train → validation

Каждая эпоха завершается сохранением состояния на Drive. Лимит эпох на сессию позволяет проверить продолжение, не меняя расписание обучения.

При улучшении validation macro-F1 обновляется `best.pt`. Для продолжения используется `last.pt`, даже если последняя эпоха уступает лучшей. В конце ячейки оценка на validation повторяется с весами лучшей эпохи.

In [13]:
def train_epoch(epoch):
    set_train_phase(net, epoch)
    loader = batches["train"]
    opt.zero_grad(set_to_none=True)
    total_loss = total_weight = 0.0
    # Сумма взвешенных потерь / сумма весов группы: эквивалент batch 16
    # также для последней неполной группы и при неравных весах классов.
    group_weight = torch.zeros((), device="cuda")
    for step, (x, y) in enumerate(loader, 1):
        y = y.cuda()
        weight = criterion.weight[y].sum()
        with torch.autocast("cuda", enabled=scaler.is_enabled()):
            logits = net(x.cuda())
            loss = criterion(logits, y)
        if not torch.isfinite(loss):
            raise ValueError("Неконечная train loss")
        # Нормировать до масштабирования AMP, чтобы градиенты оставались устойчивыми.
        group_scale = criterion.weight.max() * cfg["batch"] * cfg["accumulate"]
        scaler.scale(loss * weight / group_scale).backward()
        group_weight += weight
        if step % cfg["accumulate"] == 0 or step == len(loader):
            for param in net.parameters():
                if param.grad is not None:
                    param.grad.mul_(group_scale / group_weight)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            group_weight.zero_()
        total_loss += loss.item() * weight.item()
        total_weight += weight.item()
    return total_loss / total_weight

end = min(cfg["epochs"], len(history) + (epochs_this_session or cfg["epochs"]))
if mode != "test":
    for epoch in range(len(history) + 1, end + 1):
        if stale >= cfg["patience"]:
            break
        start = time.monotonic()
        lr = opt.param_groups[0]["lr"]
        loss = train_epoch(epoch)
        val = evaluate("validation")
        schedule.step()
        improved = val["macro_f1"] > high
        if improved:
            high, best_epoch, stale = val["macro_f1"], epoch, 0
        else:
            stale += 1
        history.append(dict(epoch=epoch, loss=loss, validation=val, lr=lr,
                            seconds=time.monotonic() - start))
        # Сначала фиксируем завершённую эпоху: из last.pt можно восстановить и новый лучший checkpoint.
        save_pt(dict(net=checkpoint_network(net).state_dict(), params=params, history=history, best_f1=high,
                     best_epoch=best_epoch, stale=stale, optimizer=opt.state_dict(),
                     schedule=schedule.state_dict(), scaler=scaler.state_dict(), rng=rng_state()), last)
        if improved:
            save_pt(dict(net=checkpoint_network(net).state_dict(), params=params, epoch=epoch, f1=high), best)
        save_json(history, run / "history.json")
        print(f"Эпоха {epoch}: loss={loss:.4f}, validation macro-F1={val['macro_f1']:.4f}")

winner = load_pt(best)
checkpoint_network(net).load_state_dict(winner["net"])
verified = evaluate("validation")
assert np.isclose(verified["macro_f1"], high) and winner["epoch"] == best_epoch
print("Лучшая эпоха:", best_epoch, "validation macro-F1:", high)

Эпоха 1: loss=1.0868, validation macro-F1=0.1406
Эпоха 2: loss=1.0374, validation macro-F1=0.3055


KeyboardInterrupt: 

## Итоговый test

Выберите режим `test` только после фиксации настроек. Готовый результат читается с Drive при повторном запуске.

Ячейка сохраняет метрики в `test.json`. Строки матрицы ошибок соответствуют истинным классам, столбцы — предсказанным; порядок классов тот же, что у выходов модели.

In [ ]:
if mode == "test":
    if len(history) < cfg["epochs"] and stale < cfg["patience"]:
        raise ValueError("Сначала завершите обучение в режиме resume.")
    result_path = run / "test.json"
    # Повторное выполнение читает сохранённые метрики этой пары модель/seed.
    if not result_path.exists():
        result = dict(model=model, seed=seed, split=digest, epoch=best_epoch,
                      training_seconds=sum(row["seconds"] for row in history),
                      metrics=evaluate("test"))
        save_json(result, result_path)
    display(json.loads(result_path.read_text()))

## Сравнение девяти запусков

Время включает обучение и validation на T4. Стандартное отклонение считается по трём seed.

Столбцы `*_delta_pp` показывают разницу с baseline в процентных пунктах. Например, рост среднего macro-F1 с 0,70 до 0,73 даёт +3 п.п. До получения всех `test.json` ячейка перечисляет недостающие запуски.

In [ ]:
records, missing = [], []
for name in configs:
    for s in (42, 43, 44):
        path = base / "runs" / name / f"seed-{s}" / "test.json"
        if not path.exists():
            missing.append(f"{name}/{s}")
            continue
        item = json.loads(path.read_text())
        assert (item["model"], item["seed"], item["split"]) == (name, s, digest)
        metrics = item["metrics"]
        records.append(dict(model=name, seed=s, accuracy=metrics["accuracy"],
                            macro_f1=metrics["macro_f1"], recall_malignant=metrics["recall_malignant"],
                            minutes=item["training_seconds"] / 60))
if missing:
    print("Ожидаются результаты:", ", ".join(missing))
else:
    frame = pd.DataFrame(records)
    summary = frame.groupby("model").agg(
        accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
        f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
        recall_malignant=("recall_malignant", "mean"), minutes_mean=("minutes", "mean"))
    for col in ("f1_mean", "recall_malignant"):
        summary[col + "_delta_pp"] = 100 * (summary[col] - summary.loc["baseline", col])
    display(summary)
    summary.to_csv(base / "comparison.csv")